In [1]:
import pandas as pd
import re
import json

In [ ]:
folder_name = "/content/drive/MyDrive/Диплом 2/код/labeled/"

file_name = "4th Infantry Division  Labels"
input_sufix = ".json"
output_sufix = " ideal processed.csv"

data_format="labelstudio_json"      # "csv" або "labelstudio_json"

In [ ]:
folder_name = "/content/drive/MyDrive/Диплом 2/код/labeled/"

file_name = "035_G3  Operations  June 1944  After Action Report"
input_sufix = " entities4.csv"
output_sufix = " processed.csv"

data_format="csv"      # "csv" або "labelstudio_json"

In [ ]:
#Імена файлів


input_full_name = folder_name+file_name+input_sufix
output_full_name = folder_name+file_name+output_sufix

# Читаємо дані
if data_format == "csv":

    df = pd.read_csv(input_full_name)

    full_text = df["full_text"].iloc[0]

    entities = df.iloc[1:].copy()

    entities = entities.sort_values("start_char")


elif data_format == "labelstudio_json":

    with open(input_full_name) as f:
        data = json.load(f)

    entities_list = []
    full_text = None

    for task in data:

        if full_text is None:
            full_text = task["data"]["text"]

        if not task["annotations"]:
            continue

        for ann in task["annotations"]:
            for r in ann["result"]:
                val = r["value"]

                entities_list.append({
                    "entity_text": val["text"],
                    "label": val["labels"][0],
                    "start_char": val["start"],
                    "end_char": val["end"]
                })

entities = pd.DataFrame(entities_list)

entities = entities.sort_values("start_char")

# =========================
# РОЗБИТТЯ НА АБЗАЦИ
# =========================

paragraphs = []
start = 0

for part in full_text.split("\n"):
    end = start + len(part)

    paragraphs.append({
        "start": start,
        "end": end,
        "text": part
    })

    start = end + 1

# Виділяємо сутності по типах
dates = entities[entities.label == "DATE"].reset_index(drop=True)
times = entities[entities.label == "TIME"].reset_index(drop=True)
orgs = entities[entities.label == "ORG"].reset_index(drop=True)
locs = entities[entities.label == "LOC"].reset_index(drop=True)

# =========================
# ПОШУК ПЕРШОГО АБЗАЦУ З TIME
# =========================

first_time_paragraph_start = None

for _, t in times.iterrows():

    for p in paragraphs:
        if p["start"] <= t.start_char <= p["end"]:
            first_time_paragraph_start = p["start"]
            break

    if first_time_paragraph_start is not None:
        break

dates = dates[dates.start_char >= first_time_paragraph_start].reset_index(drop=True)
times = times[times.start_char >= first_time_paragraph_start].reset_index(drop=True)
orgs = orgs[orgs.start_char >= first_time_paragraph_start].reset_index(drop=True)
locs = locs[locs.start_char >= first_time_paragraph_start].reset_index(drop=True)

rows = []

if len(dates) > 0:

    first_date_start = dates.iloc[0].start_char

    initial_orgs = orgs[
        (orgs.start_char >= first_time_paragraph_start) &
        (orgs.start_char < first_date_start)
    ]

    for _, org in initial_orgs.iterrows():

        nearest_loc = None

        distances = locs.start_char.apply(lambda x: abs(x - org.start_char))

        if not locs.empty:
            nearest_loc = locs.iloc[distances.values.argmin()]

        if nearest_loc is None:
            continue

        rows.append({
            "file_name": file_name,

            "unit": org.entity_text,
            "unit_start": org.start_char,
            "unit_end": org.end_char,

            "datetime": "6 June 0000",
            "datetime_start": first_time_paragraph_start,
            "datetime_end": first_time_paragraph_start,

            "location": nearest_loc.entity_text,
            "location_start": nearest_loc.start_char,
            "location_end": nearest_loc.end_char
        })

# Основний прохід по DATE
for i in range(len(dates)):

    current_date = dates.iloc[i]
    current_date_start = current_date.start_char

    if i < len(dates) - 1:
        block_end = dates.iloc[i + 1].start_char
    else:
        block_end = len(full_text)

    time_value = "0000"
    time_row = None

    for _, t in times.iterrows():
        if current_date_start < t.start_char < block_end:
            time_value = t.entity_text
            time_row = t
            break

    block_orgs = orgs[
        (orgs.start_char > current_date_start) &
        (orgs.start_char < block_end)
    ]

    for _, org in block_orgs.iterrows():

        nearest_loc = None
        search_block_index = i

        while search_block_index >= 0 and nearest_loc is None:

            block_start = dates.iloc[search_block_index].start_char

            if search_block_index < len(dates) - 1:
                prev_block_end = dates.iloc[search_block_index + 1].start_char
            else:
                prev_block_end = len(full_text)

            block_locs = locs[
                (locs.start_char > block_start) &
                (locs.start_char < prev_block_end)
            ]

            if not block_locs.empty:

                distances = block_locs.start_char.apply(
                    lambda x: abs(x - org.start_char)
                )

                nearest_loc = block_locs.iloc[distances.values.argmin()]
                if nearest_loc is None:
                  continue

            search_block_index -= 1

        rows.append({
            "file_name": file_name,

            "unit": org.entity_text,
            "unit_start": org.start_char,
            "unit_end": org.end_char,

            "datetime": f"{current_date.entity_text} {time_value}",
            "datetime_start": current_date.start_char,
            "datetime_end": current_date.end_char,

            "location": nearest_loc.entity_text if nearest_loc is not None else None,
            "location_start": nearest_loc.start_char if nearest_loc is not None else None,
            "location_end": nearest_loc.end_char if nearest_loc is not None else None
        })

# Формуємо таблицю
result_df = pd.DataFrame(rows)

# Зберігаємо
result_df.to_csv(output_full_name, index=False)

In [ ]:
!pip install geopy folium

In [ ]:
from geopy.geocoders import Nominatim
import time
import folium
from geopy.distance import geodesic

In [ ]:
geolocator = Nominatim(user_agent="normandy_analysis")

def get_coords(place):
    if pd.isna(place):
        return None, None

    try:
        loc = geolocator.geocode(place + ", Normandy, France")
        time.sleep(1)

        if loc:
            return loc.latitude, loc.longitude
        else:
            return None, None
    except:
        return None, None


result_df[["lat", "lon"]] = result_df["location"].apply(
    lambda x: pd.Series(get_coords(x))
)

In [ ]:
coords_df = result_df.dropna(subset=["lat", "lon"])

center = (coords_df["lat"].median(), coords_df["lon"].median())

filtered_df = coords_df[
    coords_df.apply(
        lambda r: geodesic(center, (r["lat"], r["lon"])).km < 80,
        axis=1
    )
]

In [ ]:
m = folium.Map(location=[49.3, -0.5], zoom_start=8)

for _, row in filtered_df.iterrows():
    if pd.notna(row["lat"]):
        folium.Marker(
          [row["lat"], row["lon"]],
          popup=f"{row['location']}<br>{row['unit']}<br>{row['datetime']}",
        ).add_to(m)

In [ ]:
m